In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [3]:
from crewai import Agent, Task, Crew

In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
load_dotenv(env_path, override=True)

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise RuntimeError(f"GROQ_API_KEY not found in {env_path} (cwd={Path.cwd()})")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen/qwen3.8-27b",
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key,
    temperature=0.7,
    max_tokens=800,
    max_retries=5,
    request_timeout=120,
)

In [5]:
from utils import get_serper_api_key

serper_api_key = get_serper_api_key()
if not serper_api_key:
    raise RuntimeError(f"SERPER_API_KEY not found in {env_path} (cwd={Path.cwd()})")

os.environ["SERPER_API_KEY"] = serper_api_key

In [7]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, FileReadTool, MDXSearchTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()
read_resume = FileReadTool("./fake-resume.md")
semantic_search_resume = MDXSearchTool(
    mdx="./fake-resume.md",
    config=dict(
        llm=dict(
            provider="groq",
            config=dict(model="llama-3.1-8b-instant"),
        ),
        embedder=dict(
            provider="huggingface",
            config=dict(model="sentence-transformers/all-MiniLM-L6-v2"),
        ),
    ),
)


In [8]:
researcher = Agent(
    role="Tech Job Researcher",
    goal="Make sure to do amazing analysis on job posting to help job applicants",
    backstory="As a Job Researcher, your prowess in "
    "navigating and extracting critical "
    "information from job postings is unmatched."
    "Your skills help pinpoint the necessary "
    "qualifications and skills sought "
    "by employers, forming the foundation for "
    "effective application tailoring.",
    tools=[scrape_tool, search_tool],
    verbose=True,
    llm=llm
)

In [10]:
profiler = Agent(
    role="Personal Profiler for Engineers",
    goal="Do increditble research on job applicants "
    "to help them stand out in the job market",
    tools=[scrape_tool, search_tool, read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "Equipped with analytical prowess, you dissect "
        "and synthesize information "
        "from diverse sources to craft comprehensive "
        "personal and professional profiles, laying the "
        "groundwork for personalized resume enhancements."
    ),
    llm=llm
)

In [11]:
resume_strategist = Agent(
    role="Resume Strategist for Engineers",
    goal="Find all the best ways to make a resume stand out in the job market.",
    tools=[scrape_tool, search_tool, read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "With a strategic mind and an eye for detail, you "
        "excel at refining resumes to highlight the most "
        "relevant skills and experiences, ensuring they "
        "resonate perfectly with the job's requirements."
    ),
    llm=llm,
)

In [12]:
interview_preparer = Agent(
    role="Engineering Interview Preparer",
    goal="Create interview questions and talking points "
    "based on the resume and job requirements",
    backstory="Your role is crucial in anticipating the dynamics of "
    "interviews. With your ability to formulate key questions "
    "and talking points, you prepare candidates for success, "
    "ensuring they can confidently address all aspects of the "
    "job they are applying for.",
    verbose=True,
    tools=[scrape_tool, search_tool, read_resume, semantic_search_resume],
    llm=llm,
)

In [14]:
research_task = Task(
    description=(
        "Analyze the job posting URL provided ({job_posting_url}) "
        "to extract key skills, experiences, and qualifications "
        "required. Use the tools to gather content and identify "
        "and categorize the requirements."
    ),
    expected_output=(
        "A structured list of job requirements, including necessary "
        "skills, qualifications, and experiences."
    ),
    agent=researcher,
    async_execution=True,
)

In [15]:
profiling_task = Task(
    description=(
        "Compile a detailed personal and professional profile "
        "using the GitHub ({github_url}) URLs, and personal write-up "
        "({personal_writeup}). Utilize tools to extract and "
        "synthesize information from these sources."
    ),
    expected_output=(
        "A comprehensive profile document that includes skills, "
        "project experiences, contributions, interests, and "
        "communication style."
    ),
    agent=profiler,
    async_execution=True,
)

In [16]:
resume_strategy_task = Task(
    description=(
        "Using the profile and job requirements obtained from "
        "previous tasks, tailor the resume to highlight the most "
        "relevant areas. Employ tools to adjust and enhance the "
        "resume content. Make sure this is the best resume even but "
        "don't make up any information. Update every section, "
        "inlcuding the initial summary, work experience, skills, "
        "and education. All to better reflrect the candidates "
        "abilities and how it matches the job posting."
    ),
    expected_output=(
        "An updated resume that effectively highlights the candidate's "
        "qualifications and experiences relevant to the job."
    ),
    agent=resume_strategist,
    output_file="tailored_resume.md",
    context=[research_task, profiling_task],
)

In [17]:
interview_preparation_task = Task(
    description=(
        "Create a set of potential interview questions and talking "
        "points based on the tailored resume and job requirements. "
        "Utilize tools to generate relevant questions and discussion "
        "points. Make sure to use these question and talking points to "
        "help the candiadte highlight the main points of the resume "
        "and how it matches the job posting."
    ),
    expected_output=(
        "A document containing key questions and talking points "
        "that the candidate should prepare for the initial interview."
    ),
    output_file="interview_materials.md",
    context=[research_task, profiling_task, resume_strategy_task],
    agent=interview_preparer,
)

In [18]:
job_application_crew = Crew(
    agents=[researcher, profiler, resume_strategist, interview_preparer],
    tasks=[
        research_task,
        profiling_task,
        resume_strategy_task,
        interview_preparation_task,
    ],
    verbose=2,
)

In [20]:
job_application_inputs = {
    "job_posting_url": "https://jobs.lever.co/AIFund/6c82e23e-d954-4dd8-a734-c0c2c5ee00f1?lever-origin=applied&lever-source%5B%5D=AI+Fund",
    "github_url": "https://github.com/joaomdmoura",
    "personal_writeup": """Noah is an accomplished Software
    Engineering Leader with 18 years of experience, specializing in
    managing remote and in-office teams, and expert in multiple
    programming languages and frameworks. He holds an MBA and a strong
    background in AI and data science. Noah has successfully led
    major tech initiatives and startups, proving his ability to drive
    innovation and growth in the tech industry. Ideal for leadership
    roles that require a strategic and innovative approach.""",
}

result = job_application_crew.kickoff(inputs=job_application_inputs)

 [DEBUG]: == Working Agent: Tech Job Researcher
 [INFO]: == Starting Task: Analyze the job posting URL provided (https://jobs.lever.co/AIFund/6c82e23e-d954-4dd8-a734-c0c2c5ee00f1?lever-origin=applied&lever-source%5B%5D=AI+Fund) to extract key skills, experiences, and qualifications required. Use the tools to gather content and identify and categorize the requirements.
 [DEBUG]: == [Tech Job Researcher] Task output: 


 [DEBUG]: == Working Agent: Personal Profiler for Engineers
 [INFO]: == Starting Task: Compile a detailed personal and professional profile using the GitHub (https://github.com/joaomdmoura) URLs, and personal write-up (Noah is an accomplished Software
    Engineering Leader with 18 years of experience, specializing in
    managing remote and in-office teams, and expert in multiple
    programming languages and frameworks. He holds an MBA and a strong
    background in AI and data science. Noah has successfully led
    major tech initiatives and startups, proving his abili

Exception in thread Thread-368 (_execute):
Traceback (most recent call last):
  File "C:\Users\PC-Kosar\AppData\Local\Python\pythoncore-3.11-64\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\PC-Kosar\AppData\Local\Python\pythoncore-3.11-64\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\PC-Kosar\Desktop\Projects\multi-ai-agent-systems-crewAI\venv\Lib\site-packages\crewai\task.py", line 182, in _execute
    result = agent.execute_task(
             ^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PC-Kosar\Desktop\Projects\multi-ai-agent-systems-crewAI\venv\Lib\site-packages\crewai\agent.py", line 221, in execute_task
    result = self.agent_executor.invoke(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PC-Kosar\Desktop\Projects\multi-ai-agent-systems-crewAI\venv\Lib\site-packages\langchain\chains\base.py", line 163, in invoke
    raise e
  File "c:\Users\PC-Kosar\Desktop\Projects\multi-ai-agent-systems



> Entering new CrewAgentExecutor chain...


2026-09-21 16:59:23,339 - 20300 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)


Thought: I need to start by understanding the current state of the resume and the specific job requirements. I will first read the candidate's current resume to see what information is available. Then, I need to find out what job they are targeting. Since the prompt mentions "job requirements obtained from previous tasks" but I don't have that context in this specific turn, I will need to check if there's a standard job description or if I need to ask the Tech Job Researcher for a typical target role. However, the most logical first step is to read the local resume file to have the base content.

Action: Read a file's content
Action Input: {} 


# Noah Williams
- Email: noah.williams@example.dev
- Phone: +44 11 111 11111

## Profile
Noah Williams is a distinguished Software Engineering Leader with an 18-year tenure in the technology industry, where he has excelled in leading both remote and in-office engineering teams. His expertise spans across software development, process innovation

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3.8-27b` in organization `org_01m2wtajwgem9r6g7bvbprbjjn` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198734, Requested 2290. Please try again in 7m22.368s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

2026-09-21 17:00:08,638 - 20300 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)
